In [ ]:
!pip install geopandas Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.6 MB/s eta 0:00:00


Исправленный код, вроде норм

In [ ]:
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, shape
from Levenshtein import distance as lev_distance
from math import sqrt
import numpy as np
import os

def load_geojson(file_path):
    """Загрузка GeoJSON файла"""
    try:
        gdf = gpd.read_file(file_path)
        if gdf.geometry.isnull().any():
            gdf = gpd.GeoDataFrame(gdf.dropna(subset=['geometry']), geometry='geometry')
        if 'name' not in gdf.columns:
            gdf['name'] = ''
        return gdf
    except Exception as e:
        print(f"Ошибка загрузки файла: {e}")
        return None

def get_coordinates(geometry):
    """Извлечение координат из геометрии"""
    if geometry is None:
        return None, None
    try:
        return geometry.x, geometry.y
    except:
        return None, None

def calculate_distance(point1, point2):
    """Вычисление расстояния между двумя точками в метрах"""
    if point1 is None or point2 is None:
        return float('inf')
    try:
        x1, y1 = point1.x, point1.y
        x2, y2 = point2.x, point2.y
        dist = sqrt((x2 - x1)**2 + (y2 - y1)**2)
        return dist
    except:
        return float('inf')

def normalize_name(name):
    """Нормализация названия для сравнения"""
    if name is None:
        return ""
    try:
        normalized = str(name).lower().strip()
        for char in [',', '.', '!', '?', '-', '_']:
            normalized = normalized.replace(char, ' ')
        normalized = ' '.join(normalized.split())
        return normalized
    except:
        return ""

def find_matching_pairs(gdf1, gdf2, max_distance=50, name_similarity_threshold=0.7):
    """
    Поиск пар соответствующих объектов с маркировкой использованных.
    Обеспечивает соответствие 1:1 - каждый объект может быть сопоставлен только один раз.
    """
    matches = []

    # Создаем копию dataset1 с валидными геометриями
    gdf1_valid = gdf1[gdf1.geometry.notnull()].copy()

    if len(gdf1_valid) == 0:
        return matches

    # Сохраняем оригинальные индексы и добавляем флаг использования
    gdf1_valid['_original_index'] = gdf1_valid.index
    gdf1_valid['_used'] = False  # Флаг для отметки использованных объектов

    # Сбрасываем индекс для удобства итераций
    gdf1_valid = gdf1_valid.reset_index(drop=True)

    # Создаем пространственный индекс для ускорения поиска ближайших объектов
    try:
        if hasattr(gdf1_valid, 'sindex'):
            spatial_index = gdf1_valid.sindex
        else:
            spatial_index = None
    except:
        spatial_index = None

    # Сортируем dataset2: сначала объекты с названиями для лучшего качества сопоставления
    gdf2_sorted = gdf2.copy()
    gdf2_sorted['_has_name'] = gdf2_sorted['name'].notna() & (gdf2_sorted['name'] != '')
    gdf2_sorted = gdf2_sorted.sort_values('_has_name', ascending=False)

    # Проходим по всем объектам dataset2
    for idx2, row2 in gdf2_sorted.iterrows():
        point2 = row2.geometry
        if point2 is None:
            continue

        name2 = normalize_name(row2.get('name', ''))
        candidate_indices = []

        # Используем пространственный индекс для быстрого поиска кандидатов в пределах max_distance
        if spatial_index is not None:
            try:
                buffered = point2.buffer(max_distance)
                # Получаем индексы объектов, чьи bounding box пересекаются с буфером
                all_candidate_idxs = list(spatial_index.intersection(buffered.bounds))
                # Фильтруем только неиспользованные объекты
                candidate_indices = [idx for idx in all_candidate_idxs
                                   if idx < len(gdf1_valid) and not gdf1_valid.iloc[idx]['_used']]
            except:
                # При ошибке используем все неиспользованные объекты
                candidate_indices = [i for i in range(len(gdf1_valid)) if not gdf1_valid.iloc[i]['_used']]
        else:
            # Без пространственного индекса проверяем все неиспользованные объекты
            candidate_indices = [i for i in range(len(gdf1_valid)) if not gdf1_valid.iloc[i]['_used']]

        # Если нет кандидатов в пределах max_distance, пропускаем этот объект
        if not candidate_indices:
            continue

        best_match = None
        best_score = 0
        best_distance = float('inf')
        best_name_similarity = 0
        best_original_index = None
        best_candidate_idx = None

        # Оцениваем каждого кандидата
        for candidate_idx in candidate_indices:
            row1 = gdf1_valid.iloc[candidate_idx]
            point1 = row1.geometry

            if point1 is None:
                continue

            # Вычисляем расстояние между точками
            dist = calculate_distance(point1, point2)
            if dist > max_distance:
                continue

            # Вычисляем схожесть названий по алгоритму Левенштейна
            name1 = normalize_name(row1.get('name', ''))
            if name1 and name2:
                max_len = max(len(name1), len(name2))
                if max_len > 0:
                    lev_dist = lev_distance(name1, name2)
                    name_similarity = 1 - (lev_dist / max_len)
                else:
                    name_similarity = 0
            else:
                # Если одно из названий отсутствует, используем базовую схожесть
                name_similarity = 0.3 if name1 or name2 else 0.5

            # Комбинированная оценка: 70% за расстояние, 30% за схожесть названий
            distance_score = 1 - min(dist / max_distance, 1.0)
            score = (0.7 * distance_score + 0.3 * name_similarity)

            # Если расстояние или схожесть имен хуже порога, обнуляем оценку
            if dist > max_distance or name_similarity < name_similarity_threshold:
                score = 0

            # Сохраняем лучшего кандидата
            if score > best_score:
                best_score = score
                best_match = row1
                best_distance = dist
                best_name_similarity = name_similarity
                best_original_index = row1['_original_index']
                best_candidate_idx = candidate_idx

        # Проверяем, соответствует ли лучший кандидат пороговым значениям
        if (best_match is not None and
            best_name_similarity >= name_similarity_threshold and
            best_distance <= max_distance):

            x1, y1 = get_coordinates(best_match.geometry)
            x2, y2 = get_coordinates(point2)

            # Сохраняем информацию о сопоставленной паре
            match_info = {
                'dataset1_index': best_original_index,
                'dataset2_index': idx2,
                'dataset1_name': best_match.get('name', ''),
                'dataset2_name': row2.get('name', ''),
                'distance_meters': best_distance,
                'name_similarity': best_name_similarity,
                'combined_score': best_score,
                'dataset1_id': best_match.get('id', ''),
                'dataset2_id': row2.get('id', ''),
                'dataset1_x': x1,
                'dataset1_y': y1,
                'dataset2_x': x2,
                'dataset2_y': y2
            }
            matches.append(match_info)

            # Помечаем объект dataset1 как использованный, чтобы исключить повторное сопоставление
            gdf1_valid.loc[best_candidate_idx, '_used'] = True

    return matches

def save_unique_features(gdf, matched_indices, output_filename, dataset_num):
    """
    Сохранение уникальных объектов (не сопоставленных).
    Атрибуты координат сохраняются с указанием источника данных (X_1, Y_1 или X_2, Y_2).
    Добавляются пустые атрибуты status, true_name, X, Y для последующего заполнения.
    """
    if len(gdf) == 0:
        return

    # Фильтруем только НЕсовпадающие объекты
    unique_gdf = gdf[~gdf.index.isin(matched_indices)].copy()

    if len(unique_gdf) == 0:
        print(f"  Все объекты dataset_{dataset_num} были сопоставлены")
        return

    # Добавляем координаты с указанием источника данных
    unique_gdf[f'X_{dataset_num}'] = unique_gdf.geometry.apply(lambda geom: geom.x if geom else None)
    unique_gdf[f'Y_{dataset_num}'] = unique_gdf.geometry.apply(lambda geom: geom.y if geom else None)

    # Добавляем пустые атрибуты для последующего заполнения
    unique_gdf['status'] = None
    unique_gdf['true_name'] = None
    unique_gdf['X'] = None
    unique_gdf['Y'] = None

    # Удаляем служебные колонки, если они есть
    columns_to_drop = ['_has_name', '_used', '_original_index']
    for col in columns_to_drop:
        if col in unique_gdf.columns:
            unique_gdf = unique_gdf.drop(columns=[col])

    # Сохраняем в CSV (без геометрии)
    csv_data = unique_gdf.copy()
    if 'geometry' in csv_data.columns:
        csv_data = csv_data.drop('geometry', axis=1)

    csv_data.to_csv(output_filename, index=True, encoding='utf-8')
    print(f"  Уникальные объекты dataset_{dataset_num}: {len(unique_gdf)} -> {output_filename}")

    # Сохраняем GeoJSON с геометрией
    geojson_filename = output_filename.replace('.csv', '.geojson')
    unique_gdf.to_file(geojson_filename, driver='GeoJSON', encoding='utf-8')

def save_matched_features(gdf1, gdf2, matches, output_filename):
    """Сохранение сопоставленных объектов с геометрией в GeoJSON"""
    if not matches:
        return

    matched_features = []

    for match in matches:
        idx1 = match['dataset1_index']
        idx2 = match['dataset2_index']

        # Добавляем объект из dataset1
        if idx1 in gdf1.index:
            feature1 = gdf1.loc[idx1].copy()
            feature1['dataset'] = 'dataset1'
            feature1['matched_with'] = idx2
            feature1['match_distance'] = match['distance_meters']
            feature1['name_similarity'] = match['name_similarity']
            matched_features.append(feature1)

        # Добавляем объект из dataset2
        if idx2 in gdf2.index:
            feature2 = gdf2.loc[idx2].copy()
            feature2['dataset'] = 'dataset2'
            feature2['matched_with'] = idx1
            feature2['match_distance'] = match['distance_meters']
            feature2['name_similarity'] = match['name_similarity']
            matched_features.append(feature2)

    if matched_features:
        matched_gdf = gpd.GeoDataFrame(matched_features, crs=gdf1.crs)
        matched_gdf.to_file(output_filename, driver='GeoJSON', encoding='utf-8')

def main():
    print("Загрузка данных...")
    gdf1 = load_geojson('dataset_1.geojson')
    gdf2 = load_geojson('dataset_2.geojson')

    if gdf1 is None or gdf2 is None:
        print("Ошибка: не удалось загрузить файлы")
        return

    print(f"Dataset 1: {len(gdf1)} объектов")
    print(f"Dataset 2: {len(gdf2)} объектов")

    # Приводим к одной системе координат
    if gdf1.crs != gdf2.crs:
        gdf2 = gdf2.to_crs(gdf1.crs)

    print("Поиск соответствующих пар...")
    matches = find_matching_pairs(gdf1, gdf2, max_distance=50, name_similarity_threshold=0.7)

    # Собираем индексы сопоставленных объектов
    matched_indices_1 = [m['dataset1_index'] for m in matches]
    matched_indices_2 = [m['dataset2_index'] for m in matches]

    print(f"\nНайдено пар: {len(matches)}")

    # Сохраняем результаты
    if matches:
        df_matches = pd.DataFrame(matches)
        df_matches.to_csv('matched_pairs.csv', index=False, encoding='utf-8')
        save_matched_features(gdf1, gdf2, matches, 'matched_features.geojson')
    else:
        print("Совпадающие пары не найдены")

    # Сохраняем уникальные объекты
    print("\nСохранение уникальных объектов...")
    save_unique_features(gdf1, matched_indices_1, 'unique_1.csv', 1)
    save_unique_features(gdf2, matched_indices_2, 'unique_2.csv', 2)

if __name__ == "__main__":
    main()

Загрузка данных...
Dataset 1: 324 объектов
Dataset 2: 336 объектов
Поиск соответствующих пар...

Найдено пар: 264

Сохранение уникальных объектов...
  Уникальные объекты dataset_1: 60 -> unique_1.csv
  Уникальные объекты dataset_2: 72 -> unique_2.csv
